# ICS 604: APPLIED DATA SCIENCE

## Introduction to Parameter Estimation
## Poisson Distribution
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

## Generative Modeling
 
**Generative models** are expressive representations of the underlying data-generating process. Rather than only learning patterns or decision boundaries, a generative model attempts to describe how the data were produced. In probabilistic terms, it models the distribution $p(x)$ (or $p(x,y)$ in supervised settings), allowing us to compute how likely a given example is under the assumed process.

Having access to a distribution’s parameters is extremely powerful. Once the parameters are known, we can answer many of the statistical questions we have studied so far, including:

1. Computing probabilities, expectations, and variances
2. Performing hypothesis testing
3. Making probabilistic predictions
4. Simulating new data from the estimated distribution

However, in real-world scenarios, we rarely have direct access to the true parameters that generated the data. The population distribution is unknown, and we only observe a finite sample drawn from it. Therefore, a central task in statistics and machine learning is **parameter estimation**: using a small sample to estimate the parameters of the assumed model. Once these parameters are estimated, the model serves as a compact and interpretable representation of the data-generating process.

In this sense, generative modeling connects directly to the core goals of statistical inference: learning about the population from limited observations and using that knowledge to make principled probabilistic statements.

## Statistical Inference 

In contrast to generative modeling — where we assume a model and describe how data are produced — statistical inference works in the opposite direction: we begin with observed data and reason upward toward the underlying population parameters that generated it. This upward reasoning process is called **statistical inference**.

  <img src="https://www.dropbox.com/scl/fi/44tqgh5uzvr0vqe8qsbsp/proba_stats_2.png?rlkey=jeq57ve2esjh6u7j962q8j1yk&st=ce2usrj4&dl=1" alt="drawing" width="400px"/>

In practice, we rarely observe an entire population. Instead, we work with a small sample drawn from a much larger population. From this limited sample, we attempt to estimate characteristics of the full population.

In this setting:

- The **sample** is the data we observe.
- The **population** is the (often unobservable) full set of possible observations generated by the underlying process.

The population represents the “ground truth” distribution we ultimately care about. However, measuring every element of a population is usually impossible — it may be too large, too expensive, or even conceptually infinite (e.g., all possible outcomes of a random process). Therefore, we use the sample as a proxy for the population. More formally:

- A statistic computed from the sample (e.g., sample mean, sample variance) serves as an estimate of a population parameter (e.g., true mean, true variance).
- The sample provides information about the underlying probability distribution.
- From that estimated distribution, we can compute probabilities of events, expectations, variances, and perform hypothesis tests.

In essence, statistical inference allows us to use observed data to estimate unknown parameters and draw conclusions about the population, even though we only have access to a limited sample rather than the full underlying distribution.

### Parameter Estimation

Given a dataset, we are typically interested in estimating the parameter or parameters of the probability distribution that generated the data. Parameter estimation is central to statistical inference because it allows us to move from raw observations to a compact mathematical description of the underlying process.

For example, suppose we randomly count the number of observed moving traffic citations within a 5-mile radius of the University of Hawaiʻi at Mānoa over 90 days, distributed across 12 months. Spreading the observations throughout the year helps reduce seasonal bias and ensures a more representative sample.

Since we are counting the number of events occurring within a fixed region and time period, the data are naturally modeled using a **Poisson distribution**, which is commonly used for count data. The Poisson distribution has a single parameter $\lambda$ that represents the average rate (mean number of events) over the specified interval.

The key question is: **How do we estimate $\lambda$ from our observed data?**

We will explore three different approaches:

1. **Bootstrap Confidence Intervals**: a resampling-based method to estimate uncertainty around the parameter.
2. **Maximum Likelihood Estimation (MLE)**: selecting the parameter value that makes the observed data most probable.
3. **Bayesian Framework**: treating the parameter as a random variable and updating prior beliefs using the observed data. 

In [ ]:
citations_data = pd.read_csv("data/citations_counts.tsv", index_col="Day")
citations_data.head()

In [ ]:
plt.figure(figsize=(10, 4))

_ = plt.hist(citations_data["Counts"], bins=8, density=True, 
             edgecolor='black', linewidth=1.2, alpha=0.5)

## The Poisson Distribution

The Poisson distribution is used to model the number of events occurring within a fixed period of time or a fixed region of space. It is particularly appropriate when events occur randomly and independently, and when we are counting how many times something happens.

Examples include:

- Number of daily accidents on the H1
- Number of people who enter a grocery store every hour
- Number of childbirths in Hawaii each day
- Number of chocolate chips in a cookie
- And many other count-based phenomena

### Properties of the Poisson Distribution

The Poisson distribution has **only one parameter**, which determines both its shape and location. This parameter is denoted by $\lambda$. A random variable $X$ is said to follow a Poisson distribution, written $X \sim Poisson(\lambda)$, if the following conditions hold:

- $X$ represents the *number of events* occurring in a fixed time period or spatial region.
  - Its possible values are countably infinite non-negative integers: $0,1,2,3,…$
- The events occur *independently*.
  - The occurrence of one event does not increase or decrease the probability of another event occurring.
- The mean and variance of the distribution are both equal to $\lambda$.

The probability mass function (pmf) of a Poisson random variable is:
$$
P(X=k) = \frac{e^{-\lambda} \lambda^k}{k!}
$$
where:
- $k=0,1,2,…$
- $\lambda \gt 0$ is the average rate of occurrence

This formula gives the probability of observing exactly $k$ events in the specified interval, given the average rate $\lambda$.

### Example

Assume that the number of customers visiting a store per hour follows a Poisson distribution with parameter $\lambda = 10$. In this setting, the random variable $X$ represents the number of customers entering the store during a given hour. Because the process is modeled as Poisson, it assumes that arrivals occur independently and at a constant average rate of 10 customers per hour. The probability distribution of $X$ captures the likelihood of observing different possible customer counts within a single hour.

In [ ]:
from scipy.stats import poisson

x = np.arange(0, 30, 1)
pmfs_x = poisson.pmf(x, 10)

plt.figure(figsize=(8, 3))
plt.plot(x, pmfs_x)
plt.xlabel("$X$")
_ = plt.ylabel("$P(X)$")

For example, the probability that $x$, for $x \in  \{0, 5, 10, 15, 20, 30, 50\}$, customers visit the store tomorrow between 3 PM and 4 PM can be computed using the Poisson probability mass function with parameter $\lambda = 10$. For each selected value of $x$, we evaluate the pmf at that point to determine the likelihood of observing exactly that many customers during the specified hour. The following Python code iterates over the chosen values and prints the corresponding probabilities:

In [ ]:
for x in [0, 5, 10, 15, 20, 30, 50]:
    print(f"The pmf of x = {x:2d} is {poisson.pmf(x, 10):1.20f}")

### Link Between the Mean and the Variance

In a Poisson distribution, both the mean and the variance are equal to the parameter $\lambda$. This has important implications: the average rate of events not only determines the central tendency of the distribution but also controls its spread. In practical terms, if we know $\lambda$, we automatically know both the expected number of events and the variability around that expectation. This property is unique to the Poisson distribution among common discrete distributions and provides a useful check when modeling count data — if the sample variance differs substantially from the sample mean, the Poisson assumption may not be appropriate.

In [ ]:
plt.figure(figsize=(10, 4))

x = np.arange(0, 40, 1)
pmfs_x = poisson.pmf(x, 10)
plt.plot(x, pmfs_x)

x = np.arange(160, 240, 1)
pmfs_x = poisson.pmf(x, 200)
plt.plot(x, pmfs_x)

x = np.arange(350, 450, 1)
pmfs_x = poisson.pmf(x, 400)
_ = plt.plot(x, pmfs_x)

### Notes About the Poisson Distribution

- **Gaussian approximation for large $\lambda$**: For values of $\lambda \gt 20$, the Poisson distribution can be approximated by a Gaussian distribution with mean $\mu=\lambda$ and standard deviation $\sigma = \sqrt{\lambda}$. This approximation simplifies calculations, particularly when summing probabilities or working with continuous methods.

- **Not all count data is Poisson**: The Poisson model assumes that the mean equals the variance. When the observed variance is larger than the mean — a phenomenon known as *overdispersion* — the data does not follow a Poisson distribution. Overdispersion often occurs when additional external factors or noise contribute to variability beyond what the Poisson model predicts. In such cases, alternative distributions, such as the Negative Binomial, may provide a better fit.